# Pipeline run history loader

Collects Fabric data pipeline runs and their activity runs from the Fabric REST API and upserts them into:

- `monitoring.pipeline_run_history` - one row per pipeline run
- `monitoring.pipeline_activity_run_history` - one row per activity run

**Before the first run:** attach the Lakehouse that holds the `monitoring` schema as this notebook's default Lakehouse, and run `pipeline_run_history_ddl.sql` once to create the tables and view.

The API only keeps the last 100 completed runs per pipeline, so run this often. Schedule the **`pipeline_run_monitor` pipeline** hourly, not this notebook: the pipeline runs it with `ALERTS_ENABLED = True` and emails the alerts. Runs older than the API's window can no longer be collected.

**Lookback:** `LOOKBACK_HOURS` limits each run to recent history. Set it to `None` for the first run to backfill everything the API still holds, then back to 2-3 hours for the schedule (keep it longer than the schedule interval so no run falls through a gap).

In [ ]:
# ---- Config (parameter cell: the pipeline_run_monitor pipeline overrides ALERTS_ENABLED) ----
WORKSPACE_IDS = [
    "f78b1a7f-a951-4a83-a0bb-c071a0451046",   ]

# None = every pipeline in the workspaces above; or a list of pipeline display names
PIPELINE_NAMES = None

# Skip pipelines whose name contains any of these (case-insensitive)
EXCLUDE_NAME_CONTAINS = ["obsolete"]

# Only collect runs that started or ended in the last N hours (runs still in progress are always
# collected). None = everything the API still holds - use None for the first, backfill run.
LOOKBACK_HOURS = 3

RUN_TABLE      = "monitoring.pipeline_run_history"
ACTIVITY_TABLE = "monitoring.pipeline_activity_run_history"
HEALTH_VIEW    = "monitoring.vw_pipeline_run_health"
ALERT_TABLE    = "monitoring.pipeline_alert_log"

# Alerts are only logged when the pipeline_run_monitor pipeline runs this notebook (it passes
# ALERTS_ENABLED = True and sends the message), so manual runs never swallow an alert.
ALERTS_ENABLED      = False
ALERT_STATUSES      = ["Failed", "CompletedWithErrors"]   # effective_status values that alert
ALERT_MAX_AGE_HOURS = 24    # ignore runs that ended longer ago, so a backfill doesn't alert on old failures

In [ ]:
# ---- API helpers ----
import html
import json
import re
import time
from datetime import datetime, timedelta, timezone

import requests
from notebookutils import mssparkutils

API = "https://api.fabric.microsoft.com/v1"
TERMINAL = {"Completed", "Failed", "Cancelled", "Deduped"}


def api(method, url, body=None, max_retries=5):
    """Call the Fabric REST API; retries on throttling (429) and 5xx."""
    for attempt in range(max_retries):
        token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")
        r = requests.request(method, url, json=body, timeout=60,
                             headers={"Authorization": f"Bearer {token}"})
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(int(r.headers.get("Retry-After", 5 * 2 ** attempt)))
            continue
        r.raise_for_status()
        return r.json() if r.content else {}
    r.raise_for_status()


def get_paged(url):
    """GET a Fabric list endpoint, following continuationUri."""
    rows = []
    while url:
        page = api("GET", url)
        rows.extend(page.get("value", []))
        url = page.get("continuationUri")
    return rows


def parse_ts(s):
    """API timestamp ('2024-06-22T06:35:00.7812154', optionally with Z) -> UTC datetime."""
    if not s:
        return None
    m = re.match(r"(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})(?:\.(\d+))?", s)
    frac = (m.group(2) or "0")[:6].ljust(6, "0")
    return datetime.fromisoformat(f"{m.group(1)}.{frac}").replace(tzinfo=timezone.utc)


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")


def in_window(job, cutoff, stored_status):
    """Started/ended after cutoff, still running, or stored as unfinished (needs its final status)."""
    if cutoff is None or job["status"] not in TERMINAL:
        return True
    if job["id"] in stored_status and stored_status[job["id"]] not in TERMINAL:
        return True
    return any(t and t >= cutoff for t in (parse_ts(job.get("startTimeUtc")), parse_ts(job.get("endTimeUtc"))))


def query_activity_runs(workspace_id, run_id, run_start):
    """All activity runs of one pipeline run (run_id = job instance id)."""
    url = f"{API}/workspaces/{workspace_id}/datapipelines/pipelineruns/{run_id}/queryactivityruns"
    now = datetime.now(timezone.utc)
    body = {
        "filters": [],
        "orderBy": [{"orderBy": "ActivityRunStart", "order": "ASC"}],
        # The API filters on last-updated time: from just before the run started until now
        "lastUpdatedAfter":  iso((run_start or now - timedelta(days=30)) - timedelta(hours=1)),
        "lastUpdatedBefore": iso(now + timedelta(hours=1)),
    }
    rows = []
    while True:
        page = api("POST", url, body)
        if isinstance(page, list):          # documented response: a bare array
            return rows + page
        rows.extend(page.get("value", []))  # paged response: {value, continuationToken}
        if not page.get("continuationToken"):
            return rows
        body["continuationToken"] = page["continuationToken"]


def run_row(ws_id, ws_name, pipe, job, now):
    start, end = parse_ts(job.get("startTimeUtc")), parse_ts(job.get("endTimeUtc"))
    fail = job.get("failureReason") or {}
    return {
        "run_id":             job["id"],
        "workspace_id":       ws_id,
        "workspace_name":     ws_name,
        "pipeline_id":        pipe["id"],
        "pipeline_name":      pipe["displayName"],
        "job_type":           job.get("jobType"),
        "invoke_type":        job.get("invokeType"),
        "status":             job["status"],
        "start_time_utc":     start,
        "end_time_utc":       end,
        "duration_ms":        int((end - start).total_seconds() * 1000) if start and end else None,
        "root_activity_id":   job.get("rootActivityId"),
        "failure_error_code": fail.get("errorCode"),
        "failure_message":    fail.get("message"),
        "failure_request_id": fail.get("requestId"),
        "raw_json":           json.dumps(job),
        "ingested_at_utc":    now,
        "updated_at_utc":     now,
    }


def activity_row(ws_id, pipe, run_id, a, now):
    err = a.get("error") or {}
    if not (err.get("errorCode") or err.get("message")):
        err = {}                            # the API returns an empty error object on success
    return {
        "activity_run_id":    a["activityRunId"],
        "run_id":             run_id,
        "workspace_id":       ws_id,
        "pipeline_id":        pipe["id"],
        "pipeline_name":      pipe["displayName"],   # the API's pipelineName is a GUID
        "activity_name":      a["activityName"],
        "activity_type":      a.get("activityType"),
        "status":             a["status"],
        "start_time_utc":     parse_ts(a.get("activityRunStart")),
        "end_time_utc":       parse_ts(a.get("activityRunEnd")),
        "duration_ms":        a.get("durationInMs"),
        "retry_attempt":      a.get("retryAttempt"),
        "iteration_hash":     a.get("iterationHash") or None,
        "error_code":         err.get("errorCode") or None,
        "error_message":      err.get("message") or None,
        "error_failure_type": err.get("failureType") or None,
        "error_target":       err.get("target") or None,
        "input_json":         json.dumps(a["input"]) if a.get("input") is not None else None,
        "output_json":        json.dumps(a["output"]) if a.get("output") is not None else None,
        "ingested_at_utc":    now,
        "updated_at_utc":     now,
    }


def alert_html(alerts, step_errors, max_runs=10, max_steps=3):
    """HTML body for the alert email. alerts: health-view rows; step_errors: run_id -> failed steps."""
    if not alerts:
        return ""

    def esc(s, n=250):
        s = s or ""
        return html.escape(s if len(s) <= n else s[:n] + "...")

    parts = [f"<p><b>Fabric pipeline alert</b>: {len(alerts)} run(s) need attention</p>"]
    for a in alerts[:max_runs]:
        started = f"{a.start_time_sgt:%Y-%m-%d %H:%M} SGT" if a.start_time_sgt else "start unknown"
        took = f", {a.duration_minutes} min" if a.duration_minutes is not None else ""
        lines = [f"<b>{esc(a.pipeline_name)}</b>: {esc(a.effective_status)} (started {started}{took})"]
        lines += [f"&bull; {esc(s.activity_name)}: {esc(s.error_message)}"
                  for s in step_errors.get(a.run_id, [])[:max_steps]]
        if a.failure_message:
            lines.append(f"Run error: {esc(a.failure_message)}")
        parts.append("<p>" + "<br>".join(lines) + "</p>")
    if len(alerts) > max_runs:
        parts.append(f"<p>...and {len(alerts) - max_runs} more (see the run health view)</p>")
    return "".join(parts)


def alert_subject(alerts, max_len=200):
    """Email subject naming the affected pipelines."""
    if not alerts:
        return ""
    names = ", ".join(sorted({a.pipeline_name for a in alerts}))
    s = f"[Fabric] {len(alerts)} pipeline run(s) need attention: {names}"
    return s if len(s) <= max_len else s[:max_len - 3] + "..."

In [ ]:
# ---- Collect from the API ----
spark.conf.set("spark.sql.session.timeZone", "UTC")
now = datetime.now(timezone.utc)
cutoff = now - timedelta(hours=LOOKBACK_HOURS) if LOOKBACK_HOURS else None
print(f"Lookback: {'all runs the API holds' if cutoff is None else f'since {cutoff:%Y-%m-%d %H:%M} UTC'}")

# Runs already stored in a final state, with their activities loaded, aren't re-queried.
stored_status  = {r.run_id: r.status for r in spark.table(RUN_TABLE).select("run_id", "status").collect()}
has_activities = {r.run_id for r in spark.table(ACTIVITY_TABLE).select("run_id").distinct().collect()}

run_rows, activity_rows, errors = [], [], []

for ws_id in WORKSPACE_IDS:
    ws_name = api("GET", f"{API}/workspaces/{ws_id}")["displayName"]
    pipelines = get_paged(f"{API}/workspaces/{ws_id}/items?type=DataPipeline")
    if PIPELINE_NAMES:
        pipelines = [p for p in pipelines if p["displayName"] in PIPELINE_NAMES]
    skipped = [p["displayName"] for p in pipelines
               if any(x.lower() in p["displayName"].lower() for x in EXCLUDE_NAME_CONTAINS)]
    pipelines = [p for p in pipelines if p["displayName"] not in skipped]
    if skipped:
        print(f"Skipped {len(skipped)} excluded pipeline(s): {', '.join(sorted(skipped))}")

    for pipe in pipelines:
        try:
            jobs = get_paged(f"{API}/workspaces/{ws_id}/items/{pipe['id']}/jobs/instances")
            jobs = [j for j in jobs if in_window(j, cutoff, stored_status)]
        except requests.HTTPError as e:
            errors.append(f"{pipe['displayName']}: {e}")
            continue

        queried = 0
        for job in jobs:
            run_rows.append(run_row(ws_id, ws_name, pipe, job, now))

            settled = (job["status"] in TERMINAL
                       and stored_status.get(job["id"]) == job["status"]
                       and job["id"] in has_activities)
            if settled or job["status"] in ("NotStarted", "Deduped"):
                continue
            try:
                for a in query_activity_runs(ws_id, job["id"], parse_ts(job.get("startTimeUtc"))):
                    activity_rows.append(activity_row(ws_id, pipe, job["id"], a, now))
                queried += 1
            except requests.HTTPError as e:
                errors.append(f"{pipe['displayName']} run {job['id']}: {e}")

        print(f"{pipe['displayName']:<55} runs={len(jobs):>3}  activity queries={queried}")

print(f"\nCollected {len(run_rows)} runs, {len(activity_rows)} activity runs, {len(errors)} errors")

In [ ]:
# ---- Upsert into the history tables ----
def stage(rows, table, key, view):
    """Temp view shaped exactly like the target table (so MERGE can INSERT *)."""
    schema = spark.table(table).schema
    df = spark.createDataFrame([tuple(r[f.name] for f in schema) for r in rows], schema)
    df.dropDuplicates([key]).createOrReplaceTempView(view)

stage(run_rows,      RUN_TABLE,      "run_id",          "stg_pipeline_runs")
stage(activity_rows, ACTIVITY_TABLE, "activity_run_id", "stg_activity_runs")

spark.sql(f"""
MERGE INTO {RUN_TABLE} AS t
USING stg_pipeline_runs AS s
   ON t.run_id = s.run_id
WHEN MATCHED AND (   t.status <> s.status
                  OR NOT (t.start_time_utc <=> s.start_time_utc)
                  OR NOT (t.end_time_utc   <=> s.end_time_utc)) THEN UPDATE SET
    status             = s.status,
    start_time_utc     = s.start_time_utc,
    end_time_utc       = s.end_time_utc,
    duration_ms        = s.duration_ms,
    failure_error_code = s.failure_error_code,
    failure_message    = s.failure_message,
    failure_request_id = s.failure_request_id,
    raw_json           = s.raw_json,
    updated_at_utc     = s.updated_at_utc
WHEN NOT MATCHED THEN INSERT *
""")

spark.sql(f"""
MERGE INTO {ACTIVITY_TABLE} AS t
USING stg_activity_runs AS s
   ON t.activity_run_id = s.activity_run_id
WHEN MATCHED AND (   t.status <> s.status
                  OR NOT (t.end_time_utc <=> s.end_time_utc)) THEN UPDATE SET
    status             = s.status,
    start_time_utc     = s.start_time_utc,
    end_time_utc       = s.end_time_utc,
    duration_ms        = s.duration_ms,
    retry_attempt      = s.retry_attempt,
    error_code         = s.error_code,
    error_message      = s.error_message,
    error_failure_type = s.error_failure_type,
    error_target       = s.error_target,
    output_json        = s.output_json,
    updated_at_utc     = s.updated_at_utc
WHEN NOT MATCHED THEN INSERT *
""")

print("MERGE done")

In [ ]:
# ---- Last 2 days ----
display(spark.sql(f"""
    SELECT pipeline_name, start_time_sgt, effective_status, duration_minutes,
           failed_activity_count, failed_activities, failure_message
    FROM {HEALTH_VIEW}
    WHERE run_date_sgt >= date_sub(current_date(), 2)
    ORDER BY start_time_sgt DESC
"""))

In [ ]:
# ---- Alerts: log newly failed runs and hand the message to the pipeline ----
if errors:
    # Partial collection: fail without logging alerts; the next run picks them up.
    raise RuntimeError("Some API calls failed:\n" + "\n".join(errors))

if not ALERTS_ENABLED:
    print("Alerts disabled (ALERTS_ENABLED = False): nothing logged or sent")
else:
    alert_since = now - timedelta(hours=ALERT_MAX_AGE_HOURS)
    statuses = ", ".join(f"'{s}'" for s in ALERT_STATUSES)
    new_alerts = spark.sql(f"""
        SELECT h.run_id, h.pipeline_name, h.effective_status, h.start_time_sgt,
               h.duration_minutes, h.failure_message
        FROM {HEALTH_VIEW} h
        JOIN {RUN_TABLE} r ON r.run_id = h.run_id
        LEFT ANTI JOIN {ALERT_TABLE} l ON l.run_id = h.run_id
        WHERE h.effective_status IN ({statuses})
          AND r.end_time_utc >= TIMESTAMP '{alert_since:%Y-%m-%d %H:%M:%S}'
        ORDER BY h.start_time_sgt
    """).collect()

    step_errors = {}
    if new_alerts:
        run_ids = ", ".join(f"'{a.run_id}'" for a in new_alerts)
        for s in spark.sql(f"""
            SELECT run_id, activity_name, error_message FROM {ACTIVITY_TABLE}
            WHERE status = 'Failed' AND run_id IN ({run_ids})
            ORDER BY start_time_utc
        """).collect():
            step_errors.setdefault(s.run_id, []).append(s)

        spark.createDataFrame([(a.run_id, a.pipeline_name, a.effective_status, now) for a in new_alerts],
                              spark.table(ALERT_TABLE).schema).write.insertInto(ALERT_TABLE)

    print(f"{len(new_alerts)} new alert(s)")
    mssparkutils.notebook.exit(json.dumps({
        "alert_count":  len(new_alerts),
        "subject":      alert_subject(new_alerts),
        "message_html": alert_html(new_alerts, step_errors),
    }))